# Fine-tune the `gemma-4-E4B-it` model for `ask`

Open this notebook in Colab, select an `A100` GPU runtime, and add `HF_TOKEN` in Colab Secrets.

In [ ]:
%pip uninstall -q -y torchao
%pip install -q 'accelerate==1.14.0' 'datasets==5.0.1' 'huggingface_hub==1.27.0' 'peft==0.20.0' 'transformers==5.14.1' 'trl==1.9.2'

In [ ]:
import importlib.metadata
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import torch
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import HfApi
from peft import LoraConfig
from transformers import AutoProcessor, Gemma4Config, Gemma4ForCausalLM, TrainerCallback
from trl import SFTConfig, SFTTrainer

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets and grant this notebook access.')

api = HfApi(token=HF_TOKEN)
account = api.whoami()

HF_NAMESPACE = os.environ.get('HF_NAMESPACE', account['name'])
BASE_MODEL = 'google/gemma-4-E4B-it'
BASE_REVISION = api.model_info(BASE_MODEL, token=HF_TOKEN).sha
print(f'Authenticated as {account["name"]}')

## Load dataset

Each run loads the latest revision of the prepared dataset from the Hugging Face Hub.

In [ ]:
DATASET_REPOSITORY = f'{HF_NAMESPACE}/gemma-4-e4b-it-ask-dataset'
print(f'Dataset: {DATASET_REPOSITORY}@main')

data = load_dataset(DATASET_REPOSITORY, token=HF_TOKEN)
train, evaluate = data['train'], data['evaluate']
SHELL_TOOLS = train[0]['tools']

print({name: len(split) for name, split in data.items()})

## Set output repositories

In [ ]:
ADAPTER_REPOSITORY = f'{HF_NAMESPACE}/gemma-4-e4b-it-ask-lora'
EXPERIMENT_NAME = 'gemma-4-e4b-it-ask-lora'
SEED = 42
RUN_DIR = Path('/content/run')
ADAPTER_DIR = Path('/content/adapter')
MERGED_MODEL_REPOSITORY = f'{HF_NAMESPACE}/gemma-4-e4b-it-ask'
MERGED_MODEL_DIR = Path('/content/merged-model')
TEMPLATE_DIRECTORY = Path('/content/ask-template')
TEMPLATE_PATH = TEMPLATE_DIRECTORY / 'gemma-4-e4b-it-ask' / 'chat_template_training.jinja'

if not TEMPLATE_PATH.is_file():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/mizydorczyk/ask.git', str(TEMPLATE_DIRECTORY)], check=True)

if not TEMPLATE_PATH.is_file():
    raise FileNotFoundError(f'Training chat template not found: {TEMPLATE_PATH}')

api.create_repo(ADAPTER_REPOSITORY, repo_type='model', private=True, exist_ok=True)
print(f'Adapter: {ADAPTER_REPOSITORY}')

In [ ]:
processor = AutoProcessor.from_pretrained(BASE_MODEL, revision=BASE_REVISION, token=HF_TOKEN)
tokenizer = processor.tokenizer
tokenizer.chat_template = TEMPLATE_PATH.read_text()
processor.chat_template = tokenizer.chat_template

example = train[0]
rendered = processor.apply_chat_template(example['messages'], tools=example['tools'], tokenize=False, add_generation_prompt=False, enable_thinking=False)
print(rendered)

## Fine-tune with LoRA

Each conversation retains its own tool declarations and Gemma's template. Loss is computed only for assistant text, tool calls, and the tool handoff marker. External tool-response bodies remain masked context.

In [ ]:
# E4B is published as a multimodal checkpoint. Load only its text tower and remap the checkpoint namespace to Gemma4ForCausalLM.
multimodal_config = Gemma4Config.from_pretrained(BASE_MODEL, revision=BASE_REVISION, token=HF_TOKEN)
model = Gemma4ForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_REVISION, token=HF_TOKEN,
    config=multimodal_config.text_config,
    key_mapping={r'^model\.language_model\.': 'model.'},
    dtype=torch.bfloat16, device_map={'': 0},
)
model.config.use_cache = False

# Keep the model and generation configs aligned with the tokenizer's special-token IDs.
for config in (model.config, model.generation_config):
    config.pad_token_id = tokenizer.pad_token_id
    config.bos_token_id = tokenizer.bos_token_id
    config.eos_token_id = tokenizer.eos_token_id

peft_config = LoraConfig(
    task_type='CAUSAL_LM', target_modules='all-linear',
    r=8, lora_alpha=32, lora_dropout=0.05, bias='none'
)

class EpochCheckCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        prompt = processor.apply_chat_template(
            [{'role': 'user', 'content': 'List files in the current directory.'}],
            tools=SHELL_TOOLS,
            tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        
        was_training = model.training
        model.eval()
        # Gemma's chat template already emits BOS, so do not add it again during tokenization.
        inputs = tokenizer(prompt, add_special_tokens=False, return_tensors='pt').to(next(model.parameters()).device)
        
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=160, do_sample=False)
        
        completion = tokenizer.decode(generated[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
        model.train(was_training)
        print(f'Epoch {state.epoch:.0f}: {completion}')
        
        return control

training_args = SFTConfig(
    output_dir=str(RUN_DIR),
    seed=SEED,
    max_length=4096,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    num_train_epochs=5,
    warmup_steps=5,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},  # Use PyTorch's current checkpointing implementation.
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,  # Restore the best saved epoch before validation and export.
    metric_for_best_model='eval_loss',  # Select the best epoch by out-of-sample loss.
    save_total_limit=2,  # Retain at most two local checkpoints on Colab's ephemeral disk.
    logging_steps=1,
    assistant_only_loss=True,  # Mask system, user, and tool-response tokens from the loss.
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=evaluate,
    processing_class=tokenizer,  # Keep TRL on its text-only path and use its built-in assistant loss mask.
    peft_config=peft_config, callbacks=[EpochCheckCallback]
)

print(f'Trainable parameters: {sum(p.numel() for p in trainer.model.parameters() if p.requires_grad):,}')

In [ ]:
unexpected_trainable = [
    name for name, parameter in trainer.model.named_parameters()
    if parameter.requires_grad and 'lora_' not in name
]
if unexpected_trainable:
    raise ValueError(f'Found trainable parameters outside LoRA adapters: {unexpected_trainable}')

multimodal_lora = [
    name for name, parameter in trainer.model.named_parameters()
    if parameter.requires_grad and any(component in name for component in ('vision_tower', 'audio_tower', 'embed_vision', 'embed_audio'))
]
if multimodal_lora:
    raise ValueError(f'Found LoRA adapters outside the language model: {multimodal_lora}')

TOOL_CALL_START_ID = tokenizer.convert_tokens_to_ids('<|tool_call>')
TOOL_RESPONSE_START_ID = tokenizer.convert_tokens_to_ids('<|tool_response>')
TOOL_RESPONSE_END_ID = tokenizer.convert_tokens_to_ids('<tool_response|>')
tool_boundary_ids = (TOOL_CALL_START_ID, TOOL_RESPONSE_START_ID, TOOL_RESPONSE_END_ID)

if any(token_id is None or token_id == tokenizer.unk_token_id for token_id in tool_boundary_ids) or len(set(tool_boundary_ids)) != len(tool_boundary_ids):
    raise ValueError('Gemma tool-boundary tokens are missing or ambiguous.')

def audit_loss_mask(dataset, split_name):
    response_blocks = 0
    
    for example_index, example in enumerate(dataset):
        input_ids, labels = example['input_ids'], example['labels']
        
        if len(input_ids) != len(labels) or not any(label != -100 for label in labels) or not any(label == -100 for label in labels):
            raise ValueError(f'{split_name}[{example_index}] has invalid assistant-only labels.')
        
        for position, token_id in enumerate(input_ids):
            if token_id == TOOL_CALL_START_ID and labels[position] == -100:
                raise ValueError(f'{split_name}[{example_index}] masks an assistant tool call.')
            
            if token_id != TOOL_RESPONSE_START_ID:
                continue
            
            response_blocks += 1
            if labels[position] == -100:
                raise ValueError(f"{split_name}[{example_index}] masks Gemma's tool handoff token.")
            
            try:
                end = input_ids.index(TOOL_RESPONSE_END_ID, position + 1)
            except ValueError as error:
                raise ValueError(f'{split_name}[{example_index}] has an unterminated tool response.') from error
            
            if any(label != -100 for label in labels[position + 1:end + 1]):
                raise ValueError(f'{split_name}[{example_index}] trains on a tool response body.')
    
    if response_blocks == 0:
        raise ValueError(f'{split_name} contains no audited tool responses.')
    
    print(f'{split_name}: assistant-only loss verified across {response_blocks} tool responses')

audit_loss_mask(trainer.train_dataset, 'train')
audit_loss_mask(trainer.eval_dataset, 'evaluate')

In [ ]:
train_result = trainer.train()
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(output_dir=str(ADAPTER_DIR))
metrics = {**train_result.metrics, **trainer.evaluate()}

best_checkpoint = trainer.state.best_model_checkpoint
best_global_step = int(Path(best_checkpoint).name.removeprefix('checkpoint-')) if best_checkpoint else None

print(metrics)

## Validate

Edit the Colab prompt field, then run the cell to render one conversation with the LoRA adapter. The shell call is only parsed and displayed; it is not executed.

In [ ]:
SYSTEM_COLOR = '34'
USER_COLOR = '34'
ASSISTANT_COLOR = '32'
FIELD_COLOR = '38;5;245'

def text_content(content):
    if not content:
        return ''
    if isinstance(content, str):
        return content
    
    return ''.join((part or {}).get('text', '') for part in content)

def render_conversation(messages, assistant_label='assistant'):
    for message in messages:
        role = message['role']
        color = {'system': SYSTEM_COLOR, 'user': USER_COLOR, 'assistant': ASSISTANT_COLOR, 'tool': ASSISTANT_COLOR}[role]
        label = assistant_label if role == 'assistant' else role
        
        content = text_content(message.get('content', ''))
        if content and role == 'tool':
            print(f'\033[{color}m[tool result]\033[0m')
            result = json.loads(content)
            
            for name, value in result.items():
                if name == 'output':
                    print(f'\033[{FIELD_COLOR}m{name}:\033[0m\n{value}')
                else:
                    print(f'\033[{FIELD_COLOR}m{name}:\033[0m {value}')
            print()
        elif content:
            print(f'\033[{color}m[{label}]\033[0m\n{content}\n')
        
        for tool_call in message.get('tool_calls') or []:
            function = tool_call['function']
            print(f"\033[{ASSISTANT_COLOR}m[{label} -> {function['name']}()]\033[0m")
            
            arguments = function.get('arguments') or {}
            arguments = json.loads(arguments) if isinstance(arguments, str) else arguments
            
            for name, value in arguments.items():
                print(f'\033[{FIELD_COLOR}m{name}:\033[0m {value}')
            print()

manual_previews = {}
validation_model = trainer.model
validation_model.eval()

prompt = 'Discard my changes to README.md.' # @param {type:"string"}
if not prompt.strip():
    raise ValueError('Enter a validation prompt.')

prompt_messages = [{'role': 'user', 'content': prompt.strip()}]
prompt = processor.apply_chat_template(prompt_messages, tools=SHELL_TOOLS, tokenize=False, add_generation_prompt=True, enable_thinking=False)
# Gemma's chat template already emits BOS, so do not add it again during tokenization.
inputs = tokenizer(prompt, add_special_tokens=False, return_tensors='pt').to(next(validation_model.parameters()).device)

with torch.inference_mode():
    generated = validation_model.generate(**inputs, max_new_tokens=160, do_sample=False)

completion = tokenizer.decode(generated[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
parsed_response = tokenizer.parse_response(completion, prefix=prompt)
manual_previews[VALIDATION_PROMPT] = {
    'prompt_messages': prompt_messages, 'tools': SHELL_TOOLS,
    'completion': completion, 'parsed_response': parsed_response
}

render_conversation(prompt_messages)
render_conversation([parsed_response], assistant_label='ask')

In [ ]:
def gpu_details():
    return {
        'name': torch.cuda.get_device_name(0),
        'total_memory_bytes': torch.cuda.get_device_properties(0).total_memory,
        'max_memory_allocated_bytes': torch.cuda.max_memory_allocated(),
        'torch_cuda_version': torch.version.cuda,
        'nvidia_smi': subprocess.check_output(
            ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total', '--format=csv,noheader,nounits'],
            text=True
        ).strip()
    }

summary = {
    'experiment': EXPERIMENT_NAME,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'base_model': BASE_MODEL,
    'base_model_class': type(model).__name__,
    'base_revision': BASE_REVISION,
    'dataset_repository': DATASET_REPOSITORY,
    'dataset_revision': 'main',
    'data': {
        'train_examples': len(train),
        'evaluate_examples': len(evaluate)
    },
    'metrics': metrics,
    'best_checkpoint': {
        'global_step': best_global_step,
        'metric_name': training_args.metric_for_best_model,
        'metric_value': trainer.state.best_metric
    },
    'loss_mask': {'assistant_only': True, 'tool_handoff_marker': 'trained', 'tool_response_bodies': 'masked'},
    'validation_manual_previews': list(manual_previews.values()),
    'gpu': gpu_details(),
    'packages': {
        name: importlib.metadata.version(name) for name in ('datasets', 'huggingface_hub', 'transformers', 'trl', 'peft', 'accelerate', 'torch')
    },
    'training': training_args.to_dict()
}

def card():
    metric_rows = '\n'.join(f'| `{name}` | {value} |' for name, value in metrics.items())
    effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
    return f'''---
base_model: {BASE_MODEL}
base_model_revision: {BASE_REVISION}
datasets:
- {DATASET_REPOSITORY}
dataset_revision: main
library_name: peft
pipeline_tag: text-generation
tags: [gemma4, peft, lora, trl, ask, text-generation]
private: true
---

# ask gemma-4-E4B-it LoRA adapter

`ask` text-only LoRA adapter for `{type(model).__name__}`.

## Training configuration

| Setting | Value |
|---|---:|
| Train examples | {len(train)} |
| Evaluate examples | {len(evaluate)} |
| Base architecture | `{type(model).__name__}` (text-only examples) |
| LoRA targets | All linear layers in the text model |
| LoRA rank | {peft_config.r} |
| LoRA alpha | {peft_config.lora_alpha} |
| LoRA dropout | {peft_config.lora_dropout} |
| Precision | BF16 |
| Maximum sequence length | {training_args.max_length} |
| Effective batch size | {effective_batch_size} |
| Epochs | {training_args.num_train_epochs:g} |
| Learning rate | {training_args.learning_rate:g} |
| Warmup steps | {training_args.warmup_steps} |
| Selected checkpoint step | {best_global_step} |
| Selected `{training_args.metric_for_best_model}` | {trainer.state.best_metric} |
| Loss | Assistant-only |

## Results

| Metric | Value |
|---|---:|
{metric_rows}

## Sources

| Source | Repository | Revision |
|---|---|---|
| Base model | [`{BASE_MODEL}`](https://huggingface.co/{BASE_MODEL}) | `{BASE_REVISION}` |
| Dataset | [`{DATASET_REPOSITORY}`](https://huggingface.co/datasets/{DATASET_REPOSITORY}) | `main` |
'''

(ADAPTER_DIR / 'validation-manual-previews.json').write_text(json.dumps(list(manual_previews.values()), indent=2) + '\n')
(ADAPTER_DIR / 'run-summary.json').write_text(json.dumps(summary, indent=2, default=str) + '\n')
(ADAPTER_DIR / 'README.md').write_text(card())

In [ ]:
adapter_commit = api.upload_folder(repo_id=ADAPTER_REPOSITORY, repo_type='model', folder_path=str(ADAPTER_DIR), allow_patterns=['adapter_config.json', 'adapter_model.safetensors', 'adapter_model.bin', 'README.md', 'run-summary.json', 'validation-manual-previews.json'], commit_message='Publish LoRA adapter and run metadata').oid

summary['adapter_revision'] = adapter_commit
(ADAPTER_DIR / 'run-summary.json').write_text(json.dumps(summary, indent=2, default=str) + '\n')

metadata_commit = api.upload_file(path_or_fileobj=str(ADAPTER_DIR / 'run-summary.json'), path_in_repo='run-summary.json', repo_id=ADAPTER_REPOSITORY, repo_type='model', commit_message='Record adapter artifact revision').oid
print(f'Adapter weights: {ADAPTER_REPOSITORY}@{adapter_commit}')
print(f'Run metadata: {ADAPTER_REPOSITORY}@{metadata_commit}')

## Merge and publish

Publish a standalone text model for vLLM.

In [ ]:
if MERGED_MODEL_DIR.exists():
    shutil.rmtree(MERGED_MODEL_DIR)

MERGED_MODEL_DIR.mkdir(parents=True)

merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_MODEL_DIR)
shutil.copyfile(TEMPLATE_PATH, MERGED_MODEL_DIR / 'chat_template.jinja')

(MERGED_MODEL_DIR / 'README.md').write_text(f'''---
base_model: {BASE_MODEL}
base_model_revision: {BASE_REVISION}
pipeline_tag: text-generation
tags: [gemma4, ask, text-only, merged-lora, text-generation]
private: true
---

# ask gemma-4-E4B-it model

Standalone text-only `Gemma4ForCausalLM` model created by merging the LoRA adapter from [`{ADAPTER_REPOSITORY}`](https://huggingface.co/{ADAPTER_REPOSITORY}) into the text tower of `{BASE_MODEL}` at `{BASE_REVISION}`.

The packaged `chat_template.jinja` is required for tool calling.
''')

api.create_repo(MERGED_MODEL_REPOSITORY, repo_type='model', private=True, exist_ok=True)
merged_commit = api.upload_folder(
    repo_id=MERGED_MODEL_REPOSITORY, repo_type='model', folder_path=str(MERGED_MODEL_DIR),
    commit_message='Publish merged text-only Gemma 4 model'
).oid

print(f'Merged deployment model: {MERGED_MODEL_REPOSITORY}@{merged_commit}')